Very proudly done with human intelligent. Signed Boonyakorn Tanrattanakorn.

In [119]:
import requests
import os

common_password = "tmp/10k-most-common.txt"

if not os.path.exists(common_password):
    with open(common_password, "wb") as f:
        response = requests.get("https://raw.githubusercontent.com/danielmiessler/SecLists/refs/heads/master/Passwords/Common-Credentials/10k-most-common.txt")
        f.write(response.content)

In [120]:
import hashlib
import bcrypt
# SHA1
m=hashlib.sha1(b"Chulalongkorn").hexdigest()
print(m)

# MD5
m=hashlib.md5(b"Chulalongkorn")
print(m)
# BCRYPT
salt = bcrypt.gensalt()
m=bcrypt.hashpw(b"Chulalongkorn", salt)
print(m)

ca8a68498ae67cd14c15f5ebf043633224005759
<md5 _hashlib.HASH object @ 0x000001A9F0A6F0F0>
b'$2b$12$1Sc7IlMg3UerhlRcECEe5uArn2kLXvysPnogkvb2XhHD5ecw5NfHq'


#

In [ ]:
# 1

target_hash = "d54cc1fe76f5186380a0939d2fc1723c44e8a5f7"
substitution = [
    ('o', '0'), 
    ('l', '1'),
    ('i', '1'),
                ]
m = hashlib.sha1()

# could make it faster by comparing hex instead of str but whatever
def check_password(passwd: str, target: str) -> bool:
    return hashlib.sha1(passwd.encode()).hexdigest() == target

def capitalize_nth(s, n):
    return s[:n] + s[n:].capitalize()

# return password string
def password_variation(out: set[str], p: str, i: int) -> None:
    if i == len(p):
        out.add(p)
        return
    # do nothing
    password_variation(out, p, i+1)
    # uppercase
    p_ = capitalize_nth(p, i)
    password_variation(out, p_, i+1)
    # substitution
    for s in substitution:
        if s[0] == p[i]:
            p_ = p[:i] + s[1] + p[i+1:]
            password_variation(out, p_, i+1)

cnt = 0
with open(common_password, 'r') as f:
    for pw in f.readlines():
        pw = pw.strip()
        variation = set()
        password_variation(variation, pw, 0)
        cnt += len(variation)
        for i in variation:
            if check_password(i, target_hash):
                print(i, target_hash)
print(f"{cnt} passwords checked!")

ThaiLanD d54cc1fe76f5186380a0939d2fc1723c44e8a5f7
2653144 passwords checked!


#

In [122]:
# 2
from time import perf_counter
T_PERIOD = 1 # s
T_WORD = "amogus".encode()
# SHA1

m=hashlib.sha1()
cnt = 0
t_start = perf_counter()
while perf_counter() - t_start <= T_PERIOD:
    m.update(T_WORD)
    cnt+=1
sha1_speed = cnt/T_PERIOD

# MD5
m=hashlib.md5()
cnt = 0
t_start = perf_counter()
while perf_counter() - t_start <= T_PERIOD:
    m.update(T_WORD)
    cnt+=1
md5_speed = cnt/T_PERIOD

# BCRYPT
salt = bcrypt.gensalt()
cnt = 0
t_start = perf_counter()
while perf_counter() - t_start <= T_PERIOD:
    bcrypt.hashpw(T_WORD, salt)
    cnt+=1
bcrypt_speed = cnt/T_PERIOD

print(f"SHA-1: {sha1_speed:.2f} h/s")
print(f"MD5: {md5_speed:.2f} h/s")
print(f"BCRYPT: {bcrypt_speed:.2f} h/s")

SHA-1: 4656937.00 h/s
MD5: 4834521.00 h/s
BCRYPT: 5.00 h/s


#
### Speed of different algorithm
- SHA-1: 4858070.00 h/s
- MD5: 4706809.00 h/s
- BCRYPT: 5.00 h/s

Let 1 year = 31,536,000 s

The combination of all password for given length is 96^n (lowercase + uppercase + digits + symbols)

To find suitable length n = log(total_hash) / log(96)

### Suitable Length
- SHA-1 can hash 153,204,095,520,000 hashes in one year. n = 7.15 = ~8 characters
- MD5 can hash 148,433,928,624,000 hashes in one year. n = 7.15 = ~8 characters
- BCRYPT can hash 157,680,000 hashes in one year. n = 4.14 = ~5 characters

#
Assume we want to crack the password in less than one week (604,800 s => 3,024,000 hashes)

We would only get to n = 3.27, which is shorter than the password length most people would use. So, it is not practical even assuming that we know the salt.

#
It is not practical since the total combination of password + salt make the table size too big. Not to mention the bcrypt algorithm being slow on purpose.

#

I would use bcrypt for the has function since it is designed to be slow to prevent brute force attack.

For each user, the password hash and a random salt is stored. So, two users with the same password wouldn't have the same hash.

Bcrypt Cost Factor of 10 will hash in 152.4 ms which is slow enough to prevent brute force but fast enough to be responsive. ([source](https://stackoverflow.com/questions/4443476/optimal-bcrypt-work-factor))

With this approach, the password should still be secured even if the password hash and salt were leaked. So there shouldn't be any problem with using normal security standard for the database.